# HapticCodec 2000-sample Colab Workflow

This notebook restores the prepared dataset from Google Drive, builds a fixed 2000-sample subset, trains the HapticCodec reconstruction stage, trains the control branch, and produces reconstruction/control outputs.

In [2]:
# Experiment configuration
REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
BRANCH = "codex/wavcaps-vae-cleanup"
REPO_DIR = "/content/thesis_hapticAE"

RUN_NAME = "hapticcodec_subset_2000"
OUTPUT_ROOT = "/content/outputs"
OUTPUT_RUN_DIR = f"{OUTPUT_ROOT}/{RUN_NAME}"

PREPARED_ARCHIVE_DRIVE = "/content/drive/MyDrive/thesis_wavcaps_as/prepared_dataset/wavcaps_haptic_prepared_as.tar"
PREPARED_RESTORE_ROOT = "/content/data"
PREPARED_DATA_DIR = f"{PREPARED_RESTORE_ROOT}/wavcaps_haptic_prepared_as"
SUBSET_ROOT = f"/content/data/subsets/{RUN_NAME}"
SUBSET_DATA_DIR = f"{SUBSET_ROOT}/data"
TRAIN_LIST_PATH = f"{SUBSET_ROOT}/train_list.txt"
VAL_LIST_PATH = f"{SUBSET_ROOT}/val_list.txt"
SUBSET_MANIFEST_PATH = f"{SUBSET_ROOT}/subset_manifest.jsonl"
RUNTIME_CONFIG_DIR = "/content/runtime_configs"
CODEC_CONFIG_PATH = f"{RUNTIME_CONFIG_DIR}/{RUN_NAME}_codec.yaml"
CONTROL_CONFIG_PATH = f"{RUNTIME_CONFIG_DIR}/{RUN_NAME}_control.yaml"
EXPORT_DIR = f"/content/drive/MyDrive/thesis_wavcaps_as/experiments/{RUN_NAME}"

SUBSET_SIZE = 2000
SUBSET_SEED = 42
TRAIN_SPLIT = 0.8

SR = 8000
T = 40000
BATCH_SIZE = 8
ANALYSIS_BATCH_SIZE = 8
CODEC_EPOCHS = 50
CONTROL_EPOCHS = 60

CHANNELS = [32, 64, 128, 128, 64]
STRIDES = [5, 4, 4, 2, 2]
FIRST_KERNEL = 25
KERNEL_SIZE = 9
RESIDUAL_KERNEL_SIZE = 7
CODE_DIM = 64
N_CODEBOOKS = 4
CODEBOOK_SIZE = 256
COMMITMENT_WEIGHT = 0.25
CONTROL_DIM = 24
CONTROL_HIDDEN = 192

LR = 2e-4
WEIGHT_DECAY = 1e-5
L1_WEIGHT = 0.4
RECON_TIME_WEIGHT = 1.0
STFT_LOSS_WEIGHT = 0.5
AMPLITUDE_WEIGHT = 1.0
ENVELOPE_WEIGHT = 1.0
STFT_LINEAR_WEIGHT = 0.02
STFT_LOG_WEIGHT = 0.02

CONTROL_WAVEFORM_L1_WEIGHT = 0.5
CONTROL_AMPLITUDE_WEIGHT = 1.0
CONTROL_ENVELOPE_WEIGHT = 1.0
CONTROL_METRIC_WEIGHT = 0.3
CONTROL_LR = 2e-4

CONTROL_METRICS = [
    "rms_energy",
    "spectral_centroid_hz",
    "spectral_flatness",
    "attack_time_s",
    "gap_ratio",
    "am_modulation_index",
    "short_term_variance",
    "envelope_entropy_bits",
]

In [3]:
# Clone repo and install deps
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git checkout "$BRANCH"
!pip install -q -r requirements.txt pyyaml soundfile librosa scikit-learn tqdm matplotlib

Cloning into '/content/thesis_hapticAE'...
remote: Enumerating objects: 972, done.
remote: Counting objects: 100% (380/380), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 972 (delta 220), reused 266 (delta 142), pack-reused 592 (from 1)
Receiving objects: 100% (972/972), 26.25 MiB | 22.55 MiB/s, done.
Resolving deltas: 100% (555/555), done.
/content/thesis_hapticAE
Branch 'codex/wavcaps-vae-cleanup' set up to track remote branch 'codex/wavcaps-vae-cleanup' from 'origin'.
Switched to a new branch 'codex/wavcaps-vae-cleanup'


In [4]:
# Mount Google Drive and restore the prepared dataset archive
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive")
Path(PREPARED_RESTORE_ROOT).mkdir(parents=True, exist_ok=True)
assert Path(PREPARED_ARCHIVE_DRIVE).exists(), f"Missing archive: {PREPARED_ARCHIVE_DRIVE}"
if not Path(PREPARED_DATA_DIR).exists():
    subprocess.run(["tar", "-xf", PREPARED_ARCHIVE_DRIVE, "-C", PREPARED_RESTORE_ROOT], check=True)
print("Prepared dataset:", PREPARED_DATA_DIR)
!find "$PREPARED_DATA_DIR" -name "*.wav" | wc -l

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Prepared dataset: /content/data/wavcaps_haptic_prepared_as
108280


In [6]:
# Build reproducible 2000-sample subset using symlinks
from pathlib import Path
import json, os, random, shutil

prepared_root = Path(PREPARED_DATA_DIR)
subset_root = Path(SUBSET_ROOT)
subset_data = Path(SUBSET_DATA_DIR)
subset_root.mkdir(parents=True, exist_ok=True)
subset_data.mkdir(parents=True, exist_ok=True)

wav_files = sorted(prepared_root.rglob("*.wav"))
assert len(wav_files) >= SUBSET_SIZE, f"Need at least {SUBSET_SIZE} wavs, found {len(wav_files)}"

rng = random.Random(SUBSET_SEED)
selected = rng.sample(wav_files, SUBSET_SIZE)
selected = sorted(selected)
train_count = int(SUBSET_SIZE * TRAIN_SPLIT)
train_files = selected[:train_count]
val_files = selected[train_count:]

for src in selected:
    rel = src.relative_to(prepared_root)
    dst = subset_data / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)
    meta_src = src.with_suffix(".am1.json")
    if meta_src.exists():
        meta_dst = dst.with_suffix(".am1.json")
        if meta_dst.exists() or meta_dst.is_symlink():
            meta_dst.unlink()
        os.symlink(meta_src, meta_dst)

with open(TRAIN_LIST_PATH, "w", encoding="utf-8") as fp:
    for path in train_files:
        fp.write(str(path) + "\n")
with open(VAL_LIST_PATH, "w", encoding="utf-8") as fp:
    for path in val_files:
        fp.write(str(path) + "\n")
with open(SUBSET_MANIFEST_PATH, "w", encoding="utf-8") as fp:
    for path in selected:
        row = {
            "wav": str(path),
            "split": "train" if path in train_files else "val",
        }
        fp.write(json.dumps(row) + "\n")

print("Subset built:", subset_root)
print("train:", len(train_files), "val:", len(val_files))

Subset built: /content/data/subsets/hapticcodec_subset_2000
train: 1600 val: 400


In [ ]:
# Quick sanity check on the subset
import json, random
from pathlib import Path
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

sample_wavs = random.sample(sorted(Path(SUBSET_DATA_DIR).rglob("*.wav")), 4)
fig, axes = plt.subplots(len(sample_wavs), 1, figsize=(14, 2.5 * len(sample_wavs)))
if len(sample_wavs) == 1:
    axes = [axes]
for ax, wav_path in zip(axes, sample_wavs):
    y, sr = sf.read(wav_path)
    ax.plot(y, linewidth=0.7)
    ax.set_title(wav_path.name)
    meta_path = wav_path.with_suffix(".am1.json")
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        print(wav_path.name, "->", meta.get("user_prompt", ""))
plt.tight_layout()
plt.show()

In [7]:
# Write runtime YAML configs so hyperparameters live in the notebook, not the repo
from pathlib import Path
import yaml

Path(RUNTIME_CONFIG_DIR).mkdir(parents=True, exist_ok=True)

base_cfg = {
    "seed": SUBSET_SEED,
    "model_type": "codec",
    "data": {
        "sr": SR,
        "T": T,
        "scale": 1.0,
        "segment_mode": "hapticgen",
        "normalize_mode": "none",
        "min_segment_ratio": 1.0,
        "clip_range": None,
        "train_split": TRAIN_SPLIT,
        "analysis_batch_size": ANALYSIS_BATCH_SIZE,
        "train_random_seek": True,
        "train_sample_with_replacement": False,
        "val_random_seek": False,
        "val_sample_with_replacement": False,
        "train_file_list": TRAIN_LIST_PATH,
        "val_file_list": VAL_LIST_PATH,
    },
    "model": {
        "channels": CHANNELS,
        "strides": STRIDES,
        "first_kernel": FIRST_KERNEL,
        "kernel_size": KERNEL_SIZE,
        "residual_kernel_size": RESIDUAL_KERNEL_SIZE,
        "activation": "leaky_relu",
        "norm": "group",
        "code_dim": CODE_DIM,
        "n_codebooks": N_CODEBOOKS,
        "codebook_size": CODEBOOK_SIZE,
        "commitment_weight": COMMITMENT_WEIGHT,
        "control_dim": CONTROL_DIM,
        "control_hidden": CONTROL_HIDDEN,
        "metric_dim": len(CONTROL_METRICS),
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs": CODEC_EPOCHS,
        "patience": 10,
        "min_delta": 1e-4,
        "early_stop_start": 8,
        "grad_clip": 1.0,
        "print_every": 5,
    },
    "optimizer": {
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
    },
    "scheduler": {
        "factor": 0.5,
        "patience": 8,
    },
    "loss": {
        "l1_weight": L1_WEIGHT,
        "stft_loss_weight": STFT_LOSS_WEIGHT,
        "amplitude_weight": AMPLITUDE_WEIGHT,
        "envelope_weight": ENVELOPE_WEIGHT,
        "clamp_range": 3.0,
        "recon_time_weight": RECON_TIME_WEIGHT,
        "stft_scales": [128, 256, 512, 1024],
        "stft_hop_lengths": [32, 64, 128, 256],
        "stft_win_lengths": [128, 256, 512, 1024],
        "stft_scale_weights": [1.0, 1.0, 0.8, 0.6],
        "stft_linear_weight": STFT_LINEAR_WEIGHT,
        "stft_log_weight": STFT_LOG_WEIGHT,
        "stft_eps": 1e-7,
    },
    "control_training": {
        "batch_size": BATCH_SIZE,
        "epochs": CONTROL_EPOCHS,
        "patience": 8,
        "min_delta": 1e-4,
        "early_stop_start": 6,
        "grad_clip": 1.0,
        "print_every": 5,
    },
    "control_optimizer": {
        "lr": CONTROL_LR,
        "weight_decay": WEIGHT_DECAY,
    },
    "control_scheduler": {
        "factor": 0.5,
        "patience": 6,
    },
    "control_loss": {
        "waveform_l1_weight": CONTROL_WAVEFORM_L1_WEIGHT,
        "amplitude_weight": CONTROL_AMPLITUDE_WEIGHT,
        "envelope_weight": CONTROL_ENVELOPE_WEIGHT,
        "metric_weight": CONTROL_METRIC_WEIGHT,
        "latent_weight": 0.6,
        "var_weight": 0.1,
        "cov_weight": 0.05,
        "var_target": 0.5,
        "clamp_range": 3.0,
        "metric_names": CONTROL_METRICS,
    },
}

with open(CODEC_CONFIG_PATH, "w", encoding="utf-8") as fp:
    yaml.safe_dump(base_cfg, fp, sort_keys=False)
with open(CONTROL_CONFIG_PATH, "w", encoding="utf-8") as fp:
    yaml.safe_dump(base_cfg, fp, sort_keys=False)

print(CODEC_CONFIG_PATH)
print(CONTROL_CONFIG_PATH)

/content/runtime_configs/hapticcodec_subset_2000_codec.yaml
/content/runtime_configs/hapticcodec_subset_2000_control.yaml


In [8]:
# Train codec
%cd "$REPO_DIR"
!python scripts/train_codec.py --config "$CODEC_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --output_root "$OUTPUT_ROOT" --run_name "$RUN_NAME"

/content/thesis_hapticAE
📂 Collecting audio files from: /content/data/wavcaps_haptic_prepared_as
   Found 108280 audio files
   Global RMS: 1.000000
   Batches: train=200, val=50
🖥️ Device: cuda
   Model: HapticCodec, params: 3,026,201
[Epoch 001/50 |   2.0%] lr=2.00e-04 | train=6.759900 | val=3.065615 | vq=2.66812
[Epoch 005/50 |  10.0%] lr=2.00e-04 | train=0.495765 | val=0.378231 | vq=0.07313
[Epoch 010/50 |  20.0%] lr=2.00e-04 | train=0.322433 | val=0.311151 | vq=0.06095
[Epoch 015/50 |  30.0%] lr=2.00e-04 | train=0.275182 | val=0.256312 | vq=0.03292
[Epoch 020/50 |  40.0%] lr=2.00e-04 | train=0.255132 | val=0.243348 | vq=0.03403
[Epoch 025/50 |  50.0%] lr=2.00e-04 | train=0.238879 | val=0.217194 | vq=0.02223
[Epoch 030/50 |  60.0%] lr=2.00e-04 | train=0.225060 | val=0.219444 | vq=0.02459
[Epoch 035/50 |  70.0%] lr=2.00e-04 | train=0.241161 | val=0.205153 | vq=0.01632
[Epoch 040/50 |  80.0%] lr=2.00e-04 | train=0.209766 | val=0.196381 | vq=0.01201
[Epoch 045/50 |  90.0%] lr=2.00e-04

In [11]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 757 bytes | 252.00 KiB/s, done.
From https://github.com/cindy-77jiayi/thesis_hapticAE
   3aa9fba..420d947  codex/wavcaps-vae-cleanup -> origin/codex/wavcaps-vae-cleanup
Updating 3aa9fba..420d947
Fast-forward
 colab/train_colab_hapticcodec_subset_2000.ipynb | 23 +++++++++++------------
 src/eval/evaluate.py                            |  8 +++++---
 2 files changed, 16 insertions(+), 15 deletions(-)


In [12]:
# Reconstruction evaluation
CODEC_CKPT = f"{OUTPUT_RUN_DIR}/codec/best_model.pt"
RECON_EVAL_DIR = f"{OUTPUT_RUN_DIR}/reconstruction_eval"
!python scripts/eval_reconstruction.py --config "$CODEC_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CODEC_CKPT" --output_dir "$RECON_EVAL_DIR" --n_samples 10

✅ Loaded checkpoint: /content/outputs/hapticcodec_subset_2000/codec/best_model.pt
Reconstruction Quality (Standard Deviation):
----------------------------------------------------------------------
  Sample  0: Orig STD=0.1312  Recon STD=0.0921  Ratio=70.24%  Orig Max=0.6328  Recon Max=0.5916  MSE=0.02801  MAE=0.10991  SpecLogL1=0.92612
  Sample  1: Orig STD=0.1443  Recon STD=0.1043  Ratio=72.29%  Orig Max=0.9453  Recon Max=0.9238  MSE=0.03488  MAE=0.12511  SpecLogL1=1.01496
  Sample  2: Orig STD=0.0958  Recon STD=0.0683  Ratio=71.29%  Orig Max=0.4766  Recon Max=0.4690  MSE=0.01475  MAE=0.07281  SpecLogL1=0.92321
  Sample  3: Orig STD=0.1750  Recon STD=0.1273  Ratio=72.76%  Orig Max=0.8516  Recon Max=0.9006  MSE=0.05262  MAE=0.17002  SpecLogL1=1.01576
  Sample  4: Orig STD=0.4152  Recon STD=0.3000  Ratio=72.25%  Orig Max=0.9375  Recon Max=0.9136  MSE=0.28624  MAE=0.42572  SpecLogL1=0.92755
  Sample  5: Orig STD=0.1414  Recon STD=0.0958  Ratio=67.79%  Orig Max=0.5234  Recon Max=0.4857  

In [10]:
# Train control branch
!python scripts/train_control.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --codec_checkpoint "$CODEC_CKPT" --output_root "$OUTPUT_ROOT" --run_name "$RUN_NAME"

✅ Loaded codec checkpoint: /content/outputs/hapticcodec_subset_2000/codec/best_model.pt
📊 Saved metric stats: /content/outputs/hapticcodec_subset_2000/control/metric_stats.json
[Control 001/40 |   2.5%] lr=2.00e-04 | train=0.960262 | val=0.871075 | metric=0.74512
[Control 005/40 |  12.5%] lr=2.00e-04 | train=0.526048 | val=0.620057 | metric=0.50304
[Control 010/40 |  25.0%] lr=2.00e-04 | train=0.479028 | val=0.583916 | metric=0.46833
[Control 015/40 |  37.5%] lr=2.00e-04 | train=0.452061 | val=0.564256 | metric=0.44870
[Control 020/40 |  50.0%] lr=2.00e-04 | train=0.449151 | val=0.579447 | metric=0.45903
[Control 025/40 |  62.5%] lr=2.00e-04 | train=0.437606 | val=0.557083 | metric=0.44163
[Control 030/40 |  75.0%] lr=2.00e-04 | train=0.435417 | val=0.562712 | metric=0.44137
[Control 035/40 |  87.5%] lr=2.00e-04 | train=0.431095 | val=0.539662 | metric=0.42483
[Control 040/40 | 100.0%] lr=2.00e-04 | train=0.427213 | val=0.541886 | metric=0.42727
✅ Best control val: 0.535304
📁 Checkpoin

In [13]:
# Extract control latents and fit PCA
CONTROL_CKPT = f"{OUTPUT_RUN_DIR}/control/best_control.pt"
PCA_DIR = f"{OUTPUT_RUN_DIR}/pca"
!python scripts/extract_controls.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CONTROL_CKPT" --output_dir "$PCA_DIR"

   Dataset size: 2000
✅ Loaded checkpoint: /content/outputs/hapticcodec_subset_2000/control/best_control.pt
Extracted vectors: shape=(2000, 16), mean=0.0549, std=1.1739, min=-4.0296, max=4.0265

PCA Results (16D → 8D):
--------------------------------------------------
  PC1: 0.4373 (43.73%)  cumulative: 43.73%
  PC2: 0.3112 (31.12%)  cumulative: 74.86%
  PC3: 0.1257 (12.57%)  cumulative: 87.43%
  PC4: 0.0844 (8.44%)  cumulative: 95.87%
  PC5: 0.0230 (2.30%)  cumulative: 98.17%
  PC6: 0.0087 (0.87%)  cumulative: 99.05%
  PC7: 0.0053 (0.53%)  cumulative: 99.57%
  PC8: 0.0019 (0.19%)  cumulative: 99.76%
--------------------------------------------------
  Total explained variance: 99.76%
  Z_pca shape: (2000, 8)

  Saved: /content/outputs/hapticcodec_subset_2000/pca/pca_pipe.pkl
  Saved: /content/outputs/hapticcodec_subset_2000/pca/controls_pca.pkl
  Saved: /content/outputs/hapticcodec_subset_2000/pca/Z_pca.npy

🏁 Control extraction complete. Outputs in: /content/outputs/hapticcodec_subs

In [14]:
# Build controls spec / sweep gallery / table
CONTROLS_DIR = f"{OUTPUT_RUN_DIR}/controls"
!python scripts/build_controls.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CONTROL_CKPT" --output_dir "$CONTROLS_DIR" --pca_dir "$PCA_DIR"

✅ Loaded checkpoint: /content/outputs/hapticcodec_subset_2000/control/best_control.pt
📦 Loading existing PCA from /content/outputs/hapticcodec_subset_2000/pca

Building control specification
Saved: /content/outputs/hapticcodec_subset_2000/controls/controls_spec.json

Generating sweep gallery (waveform + spectrogram)
  PC1: sweeping [-5.20, +3.86]
  PC2: sweeping [-3.14, +4.07]
  PC3: sweeping [-2.43, +2.07]
  PC4: sweeping [-2.56, +1.28]
  PC5: sweeping [-0.83, +0.84]
  PC6: sweeping [-0.70, +0.54]
  PC7: sweeping [-0.41, +0.48]
  PC8: sweeping [-0.28, +0.22]
  Saved: /content/outputs/hapticcodec_subset_2000/controls/pc_sweep_gallery/sweep_PC1.png
Figure(1600x1760)
  Saved: /content/outputs/hapticcodec_subset_2000/controls/pc_sweep_gallery/sweep_PC2.png
Figure(1600x1760)
  Saved: /content/outputs/hapticcodec_subset_2000/controls/pc_sweep_gallery/sweep_PC3.png
Figure(1600x1760)
  Saved: /content/outputs/hapticcodec_subset_2000/controls/pc_sweep_gallery/sweep_PC4.png
Figure(1600x1760)
  

In [15]:
# Extended validation
VALIDATION_DIR = f"{OUTPUT_RUN_DIR}/validation"
!python scripts/validate_extended.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CONTROL_CKPT" --output_dir "$VALIDATION_DIR" --controls_dir "$CONTROLS_DIR" --pca_dir "$PCA_DIR"

✅ Loaded: /content/outputs/hapticcodec_subset_2000/control/best_control.pt
📦 Loading existing PCA from /content/outputs/hapticcodec_subset_2000/pca

STEP 2: Extended sweeps (8 PCs × 21 steps × 27 metrics)
  PC1: [-5.20, +3.86]
  PC2: [-3.14, +4.07]
  PC3: [-2.43, +2.07]
  PC4: [-2.56, +1.28]
  PC5: [-0.83, +0.84]
  PC6: [-0.70, +0.54]
  PC7: [-0.41, +0.48]
  PC8: [-0.28, +0.22]
  Saved: /content/outputs/hapticcodec_subset_2000/validation/monotonicity_extended_heatmap.png
Figure(1890x600)

Extended metric bindings (excluding peak_amplitude):
  PC1: rms_energy(↑1.00), spectral_centroid_hz(↓1.00), spectral_slope(↓1.00), spectral_flatness(↓1.00)
  PC2: rms_energy(↓1.00), spectral_centroid_hz(↑1.00), spectral_flatness(↑1.00), high_freq_ratio(↑1.00)
  PC3: rms_energy(↓1.00), spectral_centroid_hz(↑1.00), spectral_slope(↑1.00), spectral_flatness(↑1.00)
  PC4: band_energy_0_150(↓1.00), band_energy_400_800(↑1.00), envelope_decay_slope_dBps(↑1.00), late_early_energy_ratio(↑1.00)
  PC5: rms_energy

In [16]:
# Save outputs and runtime metadata back to Drive
from pathlib import Path
import json, shutil

export_root = Path(EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
for src in [OUTPUT_RUN_DIR, CODEC_CONFIG_PATH, CONTROL_CONFIG_PATH, TRAIN_LIST_PATH, VAL_LIST_PATH, SUBSET_MANIFEST_PATH]:
    src_path = Path(src)
    dst_path = export_root / src_path.name
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

summary = {
    "run_name": RUN_NAME,
    "codec_checkpoint": CODEC_CKPT,
    "control_checkpoint": CONTROL_CKPT,
    "prepared_data_dir": PREPARED_DATA_DIR,
    "subset_root": SUBSET_ROOT,
    "export_dir": str(export_root),
}
with open(export_root / "run_summary.json", "w", encoding="utf-8") as fp:
    json.dump(summary, fp, indent=2)
print("Saved artifacts to:", export_root)

Saved artifacts to: /content/drive/MyDrive/thesis_wavcaps_as/experiments/hapticcodec_subset_2000
